# MEF Volume Medallion Builder

Builds the MEF Volume medallion **seed → silver → gold**, in order, as a single
job. Run as the `build_mef_volume_medallion` job (see
`resources/mef_volume.job.yml`).

**REQUIRED post-deploy step.** A `bundle deploy` only *creates* this job and the
dashboard resource; it does NOT run the SQL. The `mef_mvp_gold_*` views the
dashboard reads do not exist until this notebook runs. So after every deploy to a
fresh target you MUST run this job once; the dashboard is non-rendering until it
completes.

Single source of truth: `sql/mef_volume/00_seed.sql`, `10_silver.sql`,
`20_gold.sql`. This notebook does not duplicate any SQL — it reads those files
from their deployed Workspace Files location, sets the parameterized
catalog/schema context ONCE (from job params — the `.sql` files contain no
catalog/schema literals), and executes each file's statements in order.

catalog/schema are **parameter-only** (no literal defaults): the job binds them to
`${var.catalog}`/`${var.schema}` and the notebook fails fast if either is empty.

**Seed as-of date (optional).** Every seed row is stamped with ONE date: the
`seed_as_of_date` session variable declared in `00_seed.sql`, whose DEFAULT is
the sample report's snapshot date (the single documented date literal). Leave the
optional `seed_as_of_date` widget / job parameter EMPTY to use that default, or
set it (`YYYY-MM-DD`) to re-stamp the same verbatim numbers with another as-of
date. Silver/gold and both dashboards resolve the as-of from `MAX(business_date)`.


In [ ]:
# catalog/schema come ONLY from job parameters (bound to ${var.catalog}/
# ${var.schema} in resources/mef_volume.job.yml). No literal customer catalog/
# schema default — the widget defaults are EMPTY, and we fail fast if unset, so a
# misconfigured run can never silently build the medallion in the wrong place.
dbutils.widgets.text("catalog", "", "Catalog (required — from ${var.catalog})")
dbutils.widgets.text("schema", "", "Schema (required — from ${var.schema})")
# OPTIONAL: override the seed's as-of date (YYYY-MM-DD). Empty => the DEFAULT of
# the seed_as_of_date variable declared in sql/mef_volume/00_seed.sql.
dbutils.widgets.text("seed_as_of_date", "", "Seed as-of date override (optional, YYYY-MM-DD)")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
seed_as_of_override = dbutils.widgets.get("seed_as_of_date").strip()
if not catalog or not schema:
    raise ValueError(
        "catalog and schema are required job parameters (bound to "
        "${var.catalog}/${var.schema}); neither may be empty. Refusing to build "
        "the medallion without an explicit target."
    )

# Set the session context ONCE from the parameters. Every statement in the three
# SQL files resolves its unqualified relations against this catalog.schema — the
# SAME ${var.catalog}/${var.schema} the dashboard resource binds via
# dataset_catalog/dataset_schema. One source of truth per target.
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
print(f"Target catalog.schema: {catalog}.{schema}")

from datetime import date
if seed_as_of_override:
    # Validate up front; isoformat() below is the only thing interpolated into SQL.
    seed_as_of_override = date.fromisoformat(seed_as_of_override).isoformat()
print("Seed as-of date:", seed_as_of_override or "(00_seed.sql default)")

In [ ]:
# Locate the sql/mef_volume/*.sql files relative to THIS notebook's deployed
# Workspace path. The bundle syncs the tree, so the notebook lives at
#   <root>/files/src/build_mef_volume_medallion_nb  and the SQL at
#   <root>/files/sql/mef_volume/*.sql
# Read from Workspace Files (works on serverless). No __file__ on a local path.
import os

nb_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
files_root = os.path.dirname(os.path.dirname(nb_path))            # .../files
sql_dir = f"{files_root}/sql/mef_volume"
if not sql_dir.startswith("/Workspace"):
    sql_dir = "/Workspace" + sql_dir
print("notebook path:", nb_path)
print("sql dir:", sql_dir)

In [ ]:
import re

# Run the medallion in strict order: SEED -> SILVER -> GOLD. The .sql files carry
# NO USE/catalog/schema literals; the parameterized context set above governs all
# unqualified relations.
ORDERED_FILES = ["00_seed.sql", "10_silver.sql", "20_gold.sql"]

def split_sql_statements(sql):
    """Split on ';' but IGNORE semicolons inside single/double-quoted strings and
    inside -- line comments or /* */ block comments. Minimal, dependency-free
    scanner so a future embedded ';' can't silently mis-split a statement."""
    statements, buf = [], []
    i, n = 0, len(sql)
    in_squote = in_dquote = in_line_comment = in_block_comment = False
    while i < n:
        ch = sql[i]
        nxt = sql[i + 1] if i + 1 < n else ""
        if in_line_comment:
            buf.append(ch)
            if ch == "\n":
                in_line_comment = False
        elif in_block_comment:
            buf.append(ch)
            if ch == "*" and nxt == "/":
                buf.append(nxt); i += 2; in_block_comment = False; continue
        elif in_squote:
            buf.append(ch)
            if ch == "'":
                in_squote = False
        elif in_dquote:
            buf.append(ch)
            if ch == '"':
                in_dquote = False
        elif ch == "-" and nxt == "-":
            buf.append(ch); buf.append(nxt); i += 2; in_line_comment = True; continue
        elif ch == "/" and nxt == "*":
            buf.append(ch); buf.append(nxt); i += 2; in_block_comment = True; continue
        elif ch == "'":
            buf.append(ch); in_squote = True
        elif ch == '"':
            buf.append(ch); in_dquote = True
        elif ch == ";":
            stmt = "".join(buf).strip()
            if stmt:
                statements.append(stmt)
            buf = []
        else:
            buf.append(ch)
        i += 1
    tail = "".join(buf).strip()
    if tail:
        statements.append(tail)
    return statements

def run_sql_file(fname):
    path = f"{sql_dir}/{fname}"
    with open(path, "r") as fh:
        raw = fh.read()
    statements = split_sql_statements(raw)
    made = []
    for stmt in statements:
        spark.sql(stmt)
        # Right after 00_seed.sql declares seed_as_of_date (with its default),
        # apply the optional widget override before any INSERT reads it.
        if seed_as_of_override and re.search(
            r"DECLARE\s+OR\s+REPLACE\s+VARIABLE\s+seed_as_of_date\b", stmt, re.I
        ):
            spark.sql(f"SET VAR seed_as_of_date = DATE'{seed_as_of_override}'")
        m = re.search(r"CREATE\s+OR\s+REPLACE\s+VIEW\s+([A-Za-z0-9_]+)", stmt, re.I)
        if m:
            made.append(m.group(1))
    print(f"  {fname}: ran {len(statements)} statement(s); views: {made}")
    return made

created = []
for f in ORDERED_FILES:
    print(f"Running {f} ...")
    created += run_sql_file(f)
print(f"\nMedallion built. {len(created)} view(s): {created}")

In [ ]:
# Smoke-verify: the dashboard's gold views resolve and are queryable, and the
# CURRENT projection has exactly one coherent as-of date.
GOLD_VIEWS = [
    "mef_mvp_gold_volume_history", "mef_mvp_gold_reject_rate_history",
    "mef_mvp_gold_as_of", "mef_mvp_gold_volume",
    "mef_mvp_gold_composition", "mef_mvp_gold_disposition",
]
summary = []
for v in GOLD_VIEWS:
    cnt = spark.table(f"`{catalog}`.`{schema}`.`{v}`").count()
    summary.append((v, int(cnt)))
n_dates = spark.sql("SELECT COUNT(DISTINCT as_of_date) AS n FROM mef_mvp_gold_volume").collect()[0]["n"]
assert n_dates == 1, f"current projection must have exactly one as-of date, found {n_dates}"
print("current as-of dates in mef_mvp_gold_volume:", n_dates)
display(spark.createDataFrame(summary, schema=["gold_view", "row_count"]))